# Micro-Expression Spotting — Clip Output

Notebook utama untuk spotting.

Output:
- `/output/apex/metadata.xlsx`
- `/output/apex/dataset/...`

Struktur folder input dipertahankan pada `/output/apex/dataset`.


In [1]:
import gc
import os
import re
import shutil
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import cv2
import dlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from apex import ApexPhase, ApexSmoother
from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor
from spotting import SpottingConfig

In [2]:
config = SpottingConfig(
    dataset_root=PROJECT_ROOT / 'dataset' / 'anxiety_raw',
    output_root=PROJECT_ROOT / 'output' / 'apex',
    predictor_path=Path('./models/shape_predictor_68_face_landmarks.dat'),
    video_extensions={'.avi'},
    overwrite=False,
)

max_workers = min(4, os.cpu_count() or 1)

print(f'DATASET_ROOT = {config.dataset_root}')
print(f'OUTPUT_DATASET_ROOT = {config.output_dataset_root}')
print(f'METADATA_PATH = {config.metadata_path}')
print(f'MAX_WORKERS = {max_workers}')

DATASET_ROOT = /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/dataset/anxiety_raw
OUTPUT_DATASET_ROOT = /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/dataset
METADATA_PATH = /home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/output/apex/metadata.xlsx
MAX_WORKERS = 4


In [3]:
@dataclass(slots=True)
class VideoSample:
    video_path: Path
    relative_parent: Path
    stem: str

    @property
    def relative_video_key(self) -> Path:
        return self.relative_parent / self.stem


WORKER_DETECTOR = None
WORKER_PREDICTOR = None


def init_worker(predictor_path: str) -> None:
    global WORKER_DETECTOR, WORKER_PREDICTOR
    WORKER_DETECTOR = dlib.get_frontal_face_detector()
    WORKER_PREDICTOR = dlib.shape_predictor(predictor_path)


def natural_sort_key(value: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', value)]


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def discover_video_samples(dataset_root: Path, video_extensions: set[str]) -> list[VideoSample]:
    samples = []
    for video_path in sorted(dataset_root.rglob('*'), key=lambda p: natural_sort_key(str(p))):
        if not video_path.is_file():
            continue
        if video_path.suffix.lower() not in video_extensions:
            continue
        rel = video_path.relative_to(dataset_root)
        samples.append(VideoSample(video_path=video_path, relative_parent=rel.parent, stem=video_path.stem))
    return samples


def extract_region(image: np.ndarray, landmarks, indices: list[int]) -> np.ndarray:
    pts = [(landmarks.part(i).x, landmarks.part(i).y) for i in indices]
    xs, ys = zip(*pts)
    left = max(0, min(xs) - config.padding_x)
    top = max(0, min(ys) - config.padding_y)
    right = min(image.shape[1], max(xs) + config.padding_x)
    bottom = min(image.shape[0], max(ys) + config.padding_y)
    return image[top:bottom, left:right]


def detect_landmarks(gray: np.ndarray, detector, predictor):
    faces = detector(gray)
    if len(faces) > 0:
        return predictor(gray, faces[0])
    return None


def compute_frame_magnitude(prev_gray, prev_lm, curr_gray, curr_lm) -> float:
    roi_mags = []
    for region_name, lm_indices in config.regions.items():
        try:
            roi_prev = extract_region(prev_gray, prev_lm, lm_indices)
            roi_curr = extract_region(curr_gray, curr_lm, lm_indices)
            if roi_prev.size == 0 or roi_curr.size == 0:
                return 0.0
            target = config.target_size[region_name]
            roi_prev = cv2.resize(roi_prev, target)
            roi_curr = cv2.resize(roi_curr, target)
            poc = POC(roi_prev, roi_curr, config.block_size)
            vec = Vektor(poc.getPOC(), config.block_size)
            quad_data = Quadran(vec.getVektor()).getQuadran()
            magnitudes = [float(block[4]) for block in quad_data]
            roi_mags.append(np.mean(magnitudes) if magnitudes else 0.0)
        except Exception:
            return 0.0
    return float(np.mean(roi_mags)) if roi_mags else 0.0


def load_video_frames(video_path: Path) -> list[np.ndarray]:
    frames = []
    cap = cv2.VideoCapture(str(video_path))
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


def build_magnitude_signal(frames: list[np.ndarray], detector, predictor) -> list[float]:
    if len(frames) < 3:
        return []
    grays = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
    magnitudes = []
    prev_gray = grays[0]
    prev_lm = detect_landmarks(prev_gray, detector, predictor)
    for curr_gray in grays[1:]:
        curr_lm = detect_landmarks(curr_gray, detector, predictor)
        if prev_lm is not None and curr_lm is not None:
            mag = compute_frame_magnitude(prev_gray, prev_lm, curr_gray, curr_lm)
        else:
            mag = 0.0
        magnitudes.append(float(mag))
        prev_gray = curr_gray
        prev_lm = curr_lm
    return magnitudes


def detect_events(magnitudes: list[float]) -> tuple[list[float], list[dict], dict]:
    smoothed = ApexSmoother.smooth(magnitudes)
    signal_arr = np.asarray(smoothed, dtype=float)
    height_threshold = float(signal_arr.mean() + signal_arr.std())
    detector = ApexPhase(
        distance_threshold=config.distance_threshold,
        prominence_threshold=config.prominence_threshold,
        cutoff_ratio=config.cutoff_ratio,
    )
    apex_indices = detector.find_top_k_apex(smoothed, k=config.top_k_apex, height=height_threshold)
    phases = detector.find_phase(smoothed, apex_indices) if apex_indices else {}
    events = []
    for event_no, apex_idx in enumerate(apex_indices, start=1):
        phase = phases[apex_idx]
        duration = int(phase['end'] - phase['start'])
        if duration < config.min_phase_duration:
            continue
        events.append({
            'event_no': event_no,
            'onset_signal': int(phase['start']),
            'apex_signal': int(apex_idx),
            'offset_signal': int(phase['end']),
            'duration': duration,
        })
    window_length = ApexSmoother.calculate_window_length(len(magnitudes))
    meta = {
        'window_length': window_length,
        'polyorder': ApexSmoother.calculate_polyorder(window_length),
        'height_threshold': height_threshold,
    }
    return smoothed, events, meta


def signal_index_to_frame_index(signal_index: int) -> int:
    return int(signal_index + 1)


def save_event_clip_frames(frames: list[np.ndarray], sample_output_dir: Path, events: list[dict]) -> list[dict]:
    ensure_dir(sample_output_dir)
    rows = []
    for event in events:
        onset_frame = min(signal_index_to_frame_index(event['onset_signal']), len(frames) - 1)
        apex_frame = min(signal_index_to_frame_index(event['apex_signal']), len(frames) - 1)
        offset_frame = min(signal_index_to_frame_index(event['offset_signal']), len(frames) - 1)
        clip_dir = sample_output_dir / f'event_{onset_frame:05d}-{offset_frame:05d}'
        ensure_dir(clip_dir)
        for write_idx, frame_idx in enumerate(range(onset_frame, offset_frame + 1)):
            if frame_idx == apex_frame:
                out_path = clip_dir / f'frame_{write_idx:05d}-apex.png'
            else:
                out_path = clip_dir / f'frame_{write_idx:05d}.jpg'
            cv2.imwrite(str(out_path), frames[frame_idx])
        rows.append({
            'relative_sample': str(sample_output_dir.relative_to(config.output_dataset_root)),
            'event_no': event['event_no'],
            'onset_signal': event['onset_signal'],
            'apex_signal': event['apex_signal'],
            'offset_signal': event['offset_signal'],
            'onset_frame': onset_frame,
            'apex_frame': apex_frame,
            'offset_frame': offset_frame,
            'duration': event['duration'],
            'saved_frames': offset_frame - onset_frame + 1,
            'clip_dir': str(clip_dir.relative_to(config.output_root)),
        })
    return rows


def process_sample(sample: VideoSample) -> dict:
    detector = WORKER_DETECTOR
    predictor = WORKER_PREDICTOR
    relative_key = sample.relative_video_key
    sample_output_dir = config.output_dataset_root / relative_key

    if sample_output_dir.exists() and not config.overwrite:
        return {
            'status': 'skip',
            'relative_key': str(relative_key),
            'rows': [],
            'reason': 'exists',
        }

    if sample_output_dir.exists() and config.overwrite:
        shutil.rmtree(sample_output_dir)

    frames = load_video_frames(sample.video_path)
    if len(frames) < 3:
        return {
            'status': 'skip',
            'relative_key': str(relative_key),
            'rows': [],
            'reason': f'too few frames ({len(frames)})',
        }

    magnitudes = build_magnitude_signal(frames, detector, predictor)
    if len(magnitudes) < 5:
        return {
            'status': 'skip',
            'relative_key': str(relative_key),
            'rows': [],
            'reason': f'too few magnitudes ({len(magnitudes)})',
        }

    try:
        _smoothed, events, meta = detect_events(magnitudes)
    except ValueError as e:
        return {
            'status': 'skip',
            'relative_key': str(relative_key),
            'rows': [],
            'reason': str(e),
        }

    rows = save_event_clip_frames(frames, sample_output_dir, events)
    metadata_rows = []
    if not rows:
        metadata_rows.append({
            'relative_sample': str(relative_key),
            'source_video': str(sample.video_path),
            'event_no': None,
            'onset_signal': None,
            'apex_signal': None,
            'offset_signal': None,
            'onset_frame': None,
            'apex_frame': None,
            'offset_frame': None,
            'duration': None,
            'saved_frames': 0,
            'window_length': meta['window_length'],
            'polyorder': meta['polyorder'],
            'height_threshold': meta['height_threshold'],
            'clip_dir': None,
        })
    else:
        for row in rows:
            metadata_rows.append({
                'relative_sample': row['relative_sample'],
                'source_video': str(sample.video_path),
                'event_no': row['event_no'],
                'onset_signal': row['onset_signal'],
                'apex_signal': row['apex_signal'],
                'offset_signal': row['offset_signal'],
                'onset_frame': row['onset_frame'],
                'apex_frame': row['apex_frame'],
                'offset_frame': row['offset_frame'],
                'duration': row['duration'],
                'saved_frames': row['saved_frames'],
                'window_length': meta['window_length'],
                'polyorder': meta['polyorder'],
                'height_threshold': meta['height_threshold'],
                'clip_dir': row['clip_dir'],
            })

    del frames
    gc.collect()
    return {
        'status': 'processed',
        'relative_key': str(relative_key),
        'rows': metadata_rows,
        'reason': None,
    }

In [4]:
if not config.dataset_root.exists():
    raise FileNotFoundError(f'Dataset root not found: {config.dataset_root}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'Predictor not found: {config.predictor_path}')

ensure_dir(config.output_root)
ensure_dir(config.output_dataset_root)
samples = discover_video_samples(config.dataset_root, config.video_extensions)
print(f'Found {len(samples)} video files')
samples[:5]

Found 560 video files


[VideoSample(video_path=PosixPath('/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/dataset/anxiety_raw/after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec.avi'), relative_parent=PosixPath('after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q1'), stem='answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec'),
 VideoSample(video_path=PosixPath('/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/dataset/anxiety_raw/after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q2/answer_2_490a0a4a-d935-4509-9a4e-fb0a393be5b9_sec.avi'), relative_parent=PosixPath('after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q2'), stem='answer_2_490a0a4a-d935-4509-9a4e-fb0a393be5b9_sec'),
 VideoSample(video_path=PosixPath('/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/dataset/anxiety_raw/after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q3/answer_3_ce8ef7e8-63e6-46c1-a510-1b8269c27ce7_sec.avi'),

In [5]:
metadata_rows = []
total_samples = len(samples)
predictor_path = str(config.predictor_path)

with ProcessPoolExecutor(
    max_workers=max_workers,
    initializer=init_worker,
    initargs=(predictor_path,),
) as executor:
    futures = {executor.submit(process_sample, sample): sample for sample in samples}
    completed = 0
    for future in as_completed(futures):
        sample = futures[future]
        relative_key = sample.relative_video_key
        completed += 1
        try:
            result = future.result()
        except Exception as e:
            print(f'[{completed}/{total_samples}] fail {relative_key}: {e}')
            continue

        if result['status'] == 'skip':
            print(f"[{completed}/{total_samples}] skip {result['relative_key']}: {result['reason']}")
            continue

        print(f"[{completed}/{total_samples}] process {result['relative_key']}")
        metadata_rows.extend(result['rows'])

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_excel(config.metadata_path, index=False)
print(f'Saved metadata: {config.metadata_path}')
print(metadata_df.head())
metadata_df

[1/560] process after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q3/answer_3_ce8ef7e8-63e6-46c1-a510-1b8269c27ce7_sec
[2/560] process after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q4/answer_4_497eb431-dd00-4530-858f-e8ec710014e7_sec
[3/560] process after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q2/answer_2_490a0a4a-d935-4509-9a4e-fb0a393be5b9_sec
[4/560] process after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q5/answer_5_e1e6b3e5-a2d1-4832-a1fc-e03673252751_sec
[5/560] process after/anxiety/abdillah_agil_arbiansyah_1765270077268/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec
[6/560] process after/anxiety/aaisyah_nursalsabiil_ni_patriarti_1765170495474/q1/answer_1_15d591ce-051a-47f2-ac38-367c1e6189c7_sec
[7/560] process after/anxiety/abdillah_agil_arbiansyah_1765270077268/q3/answer_3_ce8ef7e8-63e6-46c1-a510-1b8269c27ce7_sec
[8/560] process after/anxiety/abdillah_agil_arbiansyah_1765270077268/q2/answer_2_490a0a4a-d935-4509-9

,relative_sample,source_video,event_no,onset_signal,apex_signal,offset_signal,onset_frame,apex_frame,offset_frame,duration,saved_frames,window_length,polyorder,height_threshold,clip_dir
0,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,1,93,98,103,94,99,104,10,11,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
1,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,2,175,205,212,176,206,213,37,38,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
2,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,3,223,232,238,224,233,239,15,16,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
3,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,4,255,260,266,256,261,267,11,12,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
4,after/anxiety/aaisyah_nursalsabiil_ni_patriart...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,5,273,281,292,274,282,293,19,20,17,4,0.171891,dataset/after/anxiety/aaisyah_nursalsabiil_ni_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2759,before/tidak/tora_digda_kristiawan_17651846948...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,1,18,25,48,19,26,49,30,31,31,4,0.494645,dataset/before/tidak/tora_digda_kristiawan_176...
2760,before/tidak/tora_digda_kristiawan_17651846948...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,2,134,169,201,135,170,202,67,68,31,4,0.494645,dataset/before/tidak/tora_digda_kristiawan_176...
2761,before/tidak/tora_digda_kristiawan_17651846948...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,3,432,440,468,433,441,469,36,37,31,4,0.494645,dataset/before/tidak/tora_digda_kristiawan_176...
2762,before/tidak/tora_digda_kristiawan_17651846948...,/home/ryuko/Documents/Codes/Python/Skripsi/Con...,4,490,499,500,491,500,501,10,11,31,4,0.494645,dataset/before/tidak/tora_digda_kristiawan_176...
